In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

R_EARTH_KM = 6371.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELED_PATH = "data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"
WORLDCITIES_PATH = "data/global_data/worldcities_clean.csv"

MODEL_DIR = "artifacts/geo_mlp_safety_worldpop_knn_v1"
CONFIG_PATH = os.path.join(MODEL_DIR, "safety_knn_v1_deeper_config.json")
SCALER_PATH = os.path.join(MODEL_DIR, "safety_knn_v1_deeper_scaler.pkl")
WEIGHTS_PATH = os.path.join(MODEL_DIR, "safety_knn_v1_deeper_best_weights.pt")

In [2]:
class GeoMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [3]:
with open(CONFIG_PATH, "r") as f:
    saved_config = json.load(f)

with open(SCALER_PATH, "rb") as f:
    scaler = pickle.load(f)

model = GeoMLP(
    input_dim=len(saved_config["feature_cols"]),
    hidden_dims=tuple(saved_config["hidden_dims"]),
    dropout=saved_config["dropout"],
).to(device)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

labeled = pd.read_csv(LABELED_PATH)
worldcities = pd.read_csv(WORLDCITIES_PATH)

if "country_norm" not in labeled.columns:
    labeled["country_norm"] = labeled["country"].astype(str).str.strip().str.lower()

if "country_norm" not in worldcities.columns:
    worldcities["country_norm"] = worldcities["country"].astype(str).str.strip().str.lower()

print("Loaded labeled:", labeled.shape)
print("Loaded worldcities:", worldcities.shape)
print("Feature count expected:", len(saved_config["feature_cols"]))

Loaded labeled: (509, 65)
Loaded worldcities: (47808, 7)
Feature count expected: 36


In [4]:
def normalize_country(country):
    return str(country).strip().lower()

def haversine_vec(lat0, lon0, lats, lons):
    lat0 = np.radians(lat0)
    lon0 = np.radians(lon0)
    lats = np.radians(lats)
    lons = np.radians(lons)

    dlat = lats - lat0
    dlon = lons - lon0

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat0) * np.cos(lats) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R_EARTH_KM * c

In [5]:
BASE_FEATURE_CANDIDATES = [
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

country_feature_lookup = (
    labeled.groupby("country_norm")[BASE_FEATURE_CANDIDATES]
    .median(numeric_only=True)
    .reset_index()
)

global_feature_defaults = labeled[BASE_FEATURE_CANDIDATES].median(numeric_only=True).to_dict()

def build_country_features(country):
    country_norm = normalize_country(country)
    row = country_feature_lookup[country_feature_lookup["country_norm"] == country_norm]

    features = {}
    for col in BASE_FEATURE_CANDIDATES:
        if len(row) > 0 and col in row.columns:
            val = row.iloc[0][col]
            if pd.isna(val):
                val = global_feature_defaults[col]
        else:
            val = global_feature_defaults[col]
        features[col] = float(val)

    return features

In [6]:
def build_worldpop_features(lat, lon, worldcities_df, large_city_threshold=500_000):
    wc = worldcities_df.copy()

    dists = haversine_vec(lat, lon, wc["lat"].values, wc["lon"].values)
    wc = wc.assign(dist_km=dists)

    features = {}

    for radius in [25, 50, 100]:
        sub = wc[wc["dist_km"] <= radius]

        num_cities = int(len(sub))
        sum_pop = float(sub["population"].fillna(0).sum()) if "population" in sub.columns else 0.0

        if len(sub) > 0 and "population" in sub.columns:
            grav = float(np.sum(sub["population"].fillna(0).values / ((sub["dist_km"].values ** 2) + 1e-3)))
        else:
            grav = 0.0

        features[f"num_cities_{radius}km"] = num_cities
        features[f"sum_pop_{radius}km"] = sum_pop
        features[f"pop_gravity_{radius}km"] = grav

    if "population" in wc.columns:
        large = wc[wc["population"].fillna(0) >= large_city_threshold].copy()
    else:
        large = wc.iloc[0:0].copy()

    if len(large) > 0:
        idx = large["dist_km"].idxmin()
        nearest_large = large.loc[idx]
        features["dist_to_nearest_large_city"] = float(nearest_large["dist_km"])
        features["pop_of_nearest_large_city"] = float(nearest_large["population"])
    else:
        features["dist_to_nearest_large_city"] = float(wc["dist_km"].min()) if len(wc) > 0 else 0.0
        features["pop_of_nearest_large_city"] = 0.0

    log_cols = [
        "num_cities_25km", "num_cities_50km", "num_cities_100km",
        "sum_pop_25km", "sum_pop_50km", "sum_pop_100km",
        "pop_gravity_25km", "pop_gravity_50km", "pop_gravity_100km",
        "pop_of_nearest_large_city",
    ]
    for col in log_cols:
        features[f"log1p_{col}"] = float(np.log1p(features.get(col, 0.0)))

    features["log1p_dist_to_nearest_large_city"] = float(np.log1p(features["dist_to_nearest_large_city"]))

    return features

In [7]:
def build_knn_features(lat, lon, country, labeled_df, k_list=(3, 5, 10)):
    country_norm = normalize_country(country)

    lats = labeled_df["lat"].values.astype(np.float32)
    lons = labeled_df["lon"].values.astype(np.float32)
    crimes = labeled_df["crime_index"].values.astype(np.float32)
    safeties = labeled_df["safety_index"].values.astype(np.float32)
    countries = labeled_df["country_norm"].astype(str).values

    dists = haversine_vec(lat, lon, lats, lons)
    order = np.argsort(dists)

    features = {}

    nn_idx = order[0]
    features["dist_nearest_labeled_city"] = float(dists[nn_idx])
    features["crime_nearest_labeled_city"] = float(crimes[nn_idx])
    features["safety_nearest_labeled_city"] = float(safeties[nn_idx])
    features["same_country_as_nearest_labeled"] = int(countries[nn_idx] == country_norm)

    for k in k_list:
        idx = order[:k]
        features[f"avg_crime_k{k}"] = float(np.mean(crimes[idx]))
        features[f"avg_safety_k{k}"] = float(np.mean(safeties[idx]))

    idx5 = order[:5]
    d5 = dists[idx5]
    w = 1.0 / (d5 + 1e-3)
    features["wavg_crime_k5"] = float(np.sum(crimes[idx5] * w) / np.sum(w))
    features["wavg_safety_k5"] = float(np.sum(safeties[idx5] * w) / np.sum(w))

    for band in [50, 100, 250]:
        features[f"num_labeled_within_{band}km"] = int(np.sum(dists <= band))
        features[f"log1p_num_labeled_within_{band}km"] = float(np.log1p(features[f"num_labeled_within_{band}km"]))

    same_country_mask = (countries == country_norm)
    same_country_idx = np.where(same_country_mask)[0]

    if len(same_country_idx) > 0:
        sc_dists = dists[same_country_idx]
        sc_order = same_country_idx[np.argsort(sc_dists)]
        sc_top = sc_order[: min(5, len(sc_order))]

        features["avg_crime_same_country_k5"] = float(np.mean(crimes[sc_top]))
        features["avg_safety_same_country_k5"] = float(np.mean(safeties[sc_top]))
        n_sc_250 = int(np.sum(dists[same_country_idx] <= 250))
    else:
        features["avg_crime_same_country_k5"] = float(features["avg_crime_k5"])
        features["avg_safety_same_country_k5"] = float(features["avg_safety_k5"])
        n_sc_250 = 0

    features["num_same_country_within_250km"] = n_sc_250
    features["log1p_num_same_country_within_250km"] = float(np.log1p(n_sc_250))
    features["log1p_dist_nearest_labeled_city"] = float(np.log1p(features["dist_nearest_labeled_city"]))

    return features

In [8]:
def build_feature_row(lat, lon, country, labeled_df, worldcities_df, feature_cols):
    row = {
        "lat": float(lat),
        "lon": float(lon),
    }

    row.update(build_country_features(country))
    row.update(build_worldpop_features(lat, lon, worldcities_df))
    row.update(build_knn_features(lat, lon, country, labeled_df))

    final_row = {}
    for col in feature_cols:
        final_row[col] = row.get(col, 0.0)

    return pd.DataFrame([final_row])

def predict_safety(lat, lon, country):
    feature_cols = saved_config["feature_cols"]
    x_df = build_feature_row(lat, lon, country, labeled, worldcities, feature_cols)
    x_scaled = scaler.transform(x_df[feature_cols].values.astype(np.float32))
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred = model(x_tensor).cpu().numpy().ravel()[0]

    return float(pred), x_df

In [9]:
pred, feature_row = predict_safety(
    lat=48.8566,
    lon=2.3522,
    country="France",
)

print("Predicted safety index:", pred)
print(feature_row.T.head(30))

Predicted safety index: 33.45001220703125
                                             0
lat                               4.885660e+01
lon                               2.352200e+00
crimeindex_2020                   4.679000e+01
crimeindex_2023                   5.199000e+01
safetyindex_2020                  5.321000e+01
age_0_14                          1.810000e+01
age_15_64                         6.220000e+01
age_65_plus                       2.000000e+01
population                        6.706000e+07
density_per_km2                   1.230000e+02
log1p_num_cities_50km             5.641907e+00
log1p_sum_pop_50km                1.678404e+01
log1p_pop_gravity_50km            2.301003e+01
log1p_num_cities_100km            5.743003e+00
log1p_sum_pop_100km               1.681287e+01
log1p_pop_gravity_100km           2.301003e+01
dist_to_nearest_large_city        1.111949e-02
log1p_dist_to_nearest_large_city  1.105813e-02
log1p_pop_of_nearest_large_city   1.621885e+01
dist_nearest_label

# testing

In [10]:
row = labeled.sample(1, random_state=0).iloc[0]
print(row[["city", "country", "lat", "lon", "safety_index"]])

city              Baghdad
country              Iraq
lat              33.30617
lon             44.387221
safety_index         41.1
Name: 90, dtype: object


In [11]:
pred, feat = predict_safety(
    lat=row["lat"],
    lon=row["lon"],
    country=row["country"],
)
print("True safety:", row["safety_index"])
print("Predicted safety:", pred)

True safety: 41.1
Predicted safety: 43.504547119140625


In [12]:
print(len(saved_config["feature_cols"]), saved_config["feature_cols"])
print(feat.columns.tolist())

36 ['lat', 'lon', 'crimeindex_2020', 'crimeindex_2023', 'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus', 'population', 'density_per_km2', 'log1p_num_cities_50km', 'log1p_sum_pop_50km', 'log1p_pop_gravity_50km', 'log1p_num_cities_100km', 'log1p_sum_pop_100km', 'log1p_pop_gravity_100km', 'dist_to_nearest_large_city', 'log1p_dist_to_nearest_large_city', 'log1p_pop_of_nearest_large_city', 'dist_nearest_labeled_city', 'log1p_dist_nearest_labeled_city', 'crime_nearest_labeled_city', 'safety_nearest_labeled_city', 'same_country_as_nearest_labeled', 'avg_crime_k5', 'avg_safety_k5', 'avg_crime_k10', 'avg_safety_k10', 'wavg_crime_k5', 'wavg_safety_k5', 'log1p_num_labeled_within_50km', 'log1p_num_labeled_within_100km', 'log1p_num_labeled_within_250km', 'avg_crime_same_country_k5', 'avg_safety_same_country_k5', 'log1p_num_same_country_within_250km']
['lat', 'lon', 'crimeindex_2020', 'crimeindex_2023', 'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus', 'population', 'density_per

In [14]:
train_cols = saved_config["feature_cols"]
infer_cols = feat.columns.tolist()

print(len(train_cols), len(infer_cols))
print(train_cols == infer_cols)
print([c for c in train_cols if c not in infer_cols])
print([c for c in infer_cols if c not in train_cols])

36 36
True
[]
[]


In [15]:
train_feats = labeled[saved_config["feature_cols"]]

cols_to_check = [
    "dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "log1p_sum_pop_50km",
    "log1p_pop_of_nearest_large_city",
]

print(train_feats.describe().T.loc[cols_to_check])
print(feat.T.loc[cols_to_check])

                                 count        mean         std        min  \
dist_nearest_labeled_city        509.0  182.757840  256.126075   0.000000   
crime_nearest_labeled_city       509.0   46.362299   15.450567  15.100000   
safety_nearest_labeled_city      509.0   53.637701   15.450567  16.400000   
log1p_sum_pop_50km               509.0   14.590094    1.662561   0.000000   
log1p_pop_of_nearest_large_city  509.0   14.402283    0.984156  13.124567   

                                       25%        50%         75%          max  
dist_nearest_labeled_city        13.673391  99.710938  233.040024  1991.906982  
crime_nearest_labeled_city       33.110001  46.599998   57.410000    83.599998  
safety_nearest_labeled_city      42.590000  53.400002   66.889999    84.900002  
log1p_sum_pop_50km               13.795514  14.702213   15.591238    18.094820  
log1p_pop_of_nearest_large_city  13.562230  14.177800   14.910450    17.447423  
                                         0
dist_nea

## 10 city test

In [16]:

def evaluate_v4_on_random_cities(
    labeled_df,
    n_samples=10,
    random_state=42,
):
    np.random.seed(random_state)
    sample = labeled_df.sample(n_samples, random_state=random_state).reset_index(drop=True)

    records = []
    for i, row in sample.iterrows():
        lat = float(row["lat"])
        lon = float(row["lon"])
        country = row["country"]
        true_safety = float(row["safety_index"])

        pred, feat = predict_safety(lat=lat, lon=lon, country=country)
        err = pred - true_safety
        abs_err = abs(err)

        records.append(
            {
                "city": row.get("city", ""),
                "country": country,
                "lat": lat,
                "lon": lon,
                "true_safety": true_safety,
                "pred_safety": pred,
                "error_pred_minus_true": err,
                "abs_error": abs_err,
            }
        )

    results = pd.DataFrame(records)
    mae = results["abs_error"].mean()
    print(results)
    print("\nMean absolute error over", n_samples, "cities:", mae)
    return results, mae

# Run it!!!!
results_10, mae_10 = evaluate_v4_on_random_cities(labeled, n_samples=10, random_state=0)

            city         country        lat         lon  true_safety  \
0        Baghdad            Iraq  33.306170   44.387221         41.1   
1          Malmo          Sweden  55.605293   13.000157         44.3   
2         Jaipur           India  26.915458   75.818982         64.9   
3      Karlsruhe         Germany  49.006870    8.403420         62.5   
4        Glasgow  United Kingdom  55.861155   -4.250169         54.7   
5        Rosario       Argentina -32.959361  -60.661702         24.8   
6      Coquitlam          Canada  49.284296 -122.793281         76.7   
7      Jerusalem          Israel  31.778847   35.225786         63.2   
8          Cairo           Egypt  30.044388   31.235726         50.0   
9  Yekaterinburg          Russia  56.838207   60.600789         53.6   

   pred_safety  error_pred_minus_true  abs_error  
0    43.504547               2.404547   2.404547  
1    45.749966               1.449966   1.449966  
2    66.841751               1.941751   1.941751  
3  

# Thoughts

honeslty pretty impressed with most of the results and we will move towards more accurate scores in future versions